In [ ]:
import math
import bisect
import gameLogic
from enum import Enum, auto
import os
import random
import numpy as np
from collections import deque

import vizdoom as vzd

import torch
import torch.nn as nn
import torch.optim as optim

def resetGame(showGame):
    game = vzd.DoomGame()
    config = {
        "doom_scenario_path": os.path.join(vzd.scenarios_path, "deathmatch3.wad"),
        "doom_map": "map01",
        "doom_skill": 2,

        "available_buttons": [
            vzd.Button.ATTACK,
            vzd.Button.MOVE_RIGHT,
            vzd.Button.MOVE_LEFT,
            vzd.Button.MOVE_BACKWARD,
            vzd.Button.MOVE_FORWARD,
            vzd.Button.TURN_RIGHT,
            vzd.Button.TURN_LEFT,
            vzd.Button.SELECT_WEAPON3,
            vzd.Button.SELECT_WEAPON4
        ],

        # MUST enable richer state
        "available_game_variables": [
            vzd.GameVariable.HEALTH,
            vzd.GameVariable.ARMOR,
            vzd.GameVariable.POSITION_X,
            vzd.GameVariable.POSITION_Y,
            vzd.GameVariable.ANGLE,
            vzd.GameVariable.KILLCOUNT,
            vzd.GameVariable.SELECTED_WEAPON,
            vzd.GameVariable.AMMO3,
            vzd.GameVariable.AMMO4
        ],

        "episode_timeout": 4200,
        "episode_start_time": 1,
        "living_reward": 0.01,
        "mode": vzd.Mode.PLAYER,
        "screen_resolution": vzd.ScreenResolution.RES_1920X1080,
        "render_hud": True,
        
    }

    game.set_config(config)
    game.set_objects_info_enabled(True)
    game.set_window_visible(showGame)

    game.init()
    return game

game = resetGame(False)

numberGenerations = 5
maxDepth = 6
treesPerGeneration = 30
topNumToKeep = 20
topKeptSubtrees = [0, 6, 5, 4, 4, 3]
episodesPerTree = 30
chanceOfEndingEarly = 0.12

crossPollinationsMapping = [
# d0 d1 d2 d3 d4 d5
 [0, 3, 1, 0, 0, 0],  # depth 0
 [ 0,10, 3, 1, 0, 0],  # depth 1
 [ 0, 0,10, 3, 1, 0],  # depth 2
 [ 0, 0, 0,10, 3, 1],  # depth 3
 [ 0, 0, 0, 0,10, 3],  # depth 4
 [ 0, 0, 0, 0, 0,10],  # depth 5
]

remainingTrees = []

allConditions = {
    "lowHealth?": gameLogic.lowHealthCondition,
    "mediumHealth?": gameLogic.mediumHealthCondition,
    "highHealth?": gameLogic.highHealthCondition,
    "lowArmor?": gameLogic.lowArmorCondition,
    "mediumArmor?": gameLogic.mediumArmorCondition,
    "highArmor?": gameLogic.highArmorCondition,
    "lowAmmoCurrent?": gameLogic.lowAmmoCurrent,
    "chaingunEquipped?": gameLogic.wieldingChaingun,
    "nearbyEnemy?": gameLogic.nearbyEnemy,
    "recentlyHurt?": gameLogic.recentlyHurt,
    "healthNearby?": gameLogic.healthNearby,
    "ammo3Nearby?": gameLogic.ammo3Nearby,
    "ammo4Nearby?": gameLogic.ammo4Nearby,
    "armorNearby?": gameLogic.armorNearby,
    "manyEnemies?": gameLogic.manyEnemies,
    "noEnemies?":gameLogic.noEnemies,
    "someRangedEnemy?": gameLogic.someRangedEnemies,
    "lowTimeRemaining?": gameLogic.lowTimeRemaining
}

allActions = {
    "switchWeapon": gameLogic.SwitchWeapon,
    "fireAndStrafe": gameLogic.FireAndStrafe,
    "directlyFlee": gameLogic.DirectlyFlee,
    "goToHealth": gameLogic.GoToHealth,
    "goToAmmo": gameLogic.GoToAmmo,
    "goToArmor": gameLogic.GoToArmor,
    "moveRandom": gameLogic.MoveRandom,
    "runAway": gameLogic.RunAway,
    "chargeIn": gameLogic.ChargeIn
}

class NodeType(Enum):
    ACTION = auto()
    CONDITION = auto()

class Node():
    def __init__(self, depth, parent, nodeType, overallTree, name, mainHandler):
        self.depth = depth
        self.parent = parent
        self.type = nodeType
        self.protected = False
        self.tree = overallTree

        self.left = None
        self.right = None

        self.nodeName = name
        self.nodeHandler = mainHandler

    def mutate(self):
        while True:
            chosenList = list(allConditions.keys()) if self.type == NodeType.CONDITION else list(allActions.keys())

            chosenIndex = random.randint(0, len(chosenList))
            newHandlerName = chosenList[chosenIndex]

            if(newHandlerName == self.nodeName):
                continue
            
            self.nodeName = newHandlerName
            self.nodeHandler = allConditions[self.nodeName] if self.type == NodeType.CONDITION else allActions[self.nodeName]
            break
            
        


class Tree():
    def __init__(self, maxDepth, probabilityOfEndingEarly):
        self.maxDepth = maxDepth
        self.probabilityOfEndingEarly = probabilityOfEndingEarly
        self.selectedAction = None
        self.allNodes = set()


    def buildNode(self, conditions, actions, currentDepth, parentNode, conditionsInPath):
        randomProbability = random.random()
        if(currentDepth == self.maxDepth or (currentDepth != 0 and randomProbability <= self.probabilityOfEndingEarly)):
            nodeType = NodeType.ACTION
        else:
            nodeType = NodeType.CONDITION
        

        chosenListToSelect = conditions if nodeType == NodeType.CONDITION else actions
        chosenElement = None
        while True:
            chosenElement = random.choice(list(chosenListToSelect.keys()))
            if(chosenElement not in conditionsInPath):
                break
        newNode = Node(currentDepth, parentNode, nodeType, self, chosenElement, chosenListToSelect[chosenElement])
        
        if(nodeType == NodeType.CONDITION):
            conditionsInPath.add(chosenElement)
            newNode.left = self.buildNode(conditions, actions, currentDepth + 1, newNode, conditionsInPath.copy())
            newNode.right = self.buildNode(conditions, actions, currentDepth + 1, newNode, conditionsInPath.copy())

        self.allNodes.add(newNode)

        return newNode    

    def decideNewAction(self, state, currentTick, currentNode):
        if(currentNode.type == NodeType.ACTION):
            self.currentAction = currentNode
            self.currentAction.nodeHandler.activateAction()
            return
        
        if(currentNode.nodeHandler(state, currentTick)):
            return self.decideNewAction(state, currentTick, currentNode.left)

        return self.decideNewAction(state, currentTick, currentNode.right)

    def updateActionTick(self, state):
        action = self.currentAction.nodeHandler.updateTickAndReturnAction(state)
        if(self.currentAction.nodeHandler.finished):
            self.selectedAction = None
        return action    

def BuildRandomTree(maxDepth, probEndingEarly, conditions, actions):
    tree = Tree(maxDepth, probEndingEarly)
    tree.root = tree.buildNode(conditions, actions, 0, None, set())
    return tree

def RankTreesByScore(treeArray):
    return sorted(treeArray, key = lambda x : x.score)

def clearNodesRecursively(currentNode):
    currentNode.protected = False
    
    if(currentNode.type == NodeType.ACTION):
        return
    clearNodesRecursively(currentNode.left)
    clearNodesRecursively(currentNode.right)

def DoTreeUpkeep(givenTree):
    tree.score = 0
    clearNodesRecursively(tree.root)

def ShowTreeStructure(node, indent=""):
    if node.type == NodeType.ACTION:
        print(indent + "[ACTION] " + node.nodeName)
        return

    print(indent + "[COND] " + node.nodeName)

    print(indent + "  T:")
    ShowTreeStructure(node.left, indent + "    ")

    print(indent + "  F:")
    ShowTreeStructure(node.right, indent + "    ")

def RunGame(behaviorTree, game, episodes):
    global previousHealth
    score = 0
    for episode in range(episodes):
        game.new_episode()

        state = game.get_state()

        tick = 0

        epScore = 0
        while not game.is_episode_finished():
            if behaviorTree.selectedAction is None:
                behaviorTree.decideNewAction(state, tick, behaviorTree.root)

            chosenAction = behaviorTree.updateActionTick(state)

            epScore += game.make_action(chosenAction) / 100.0

            health, armor, posX, posY, angle, kills, currentWeapon, firstWepAmmo, secondWepAmmo = state.game_variables

            if(health > gameLogic.previousHealth):
                gameLogic.previousHealth = health

            state = game.get_state()

            tick += 1

        score += epScore / episodes

    return score


numTreesForReproduction = 0.05 * treesPerGeneration
numTreesForCrossover = 0.75 * treesPerGeneration
numTreesForMutation = 0.15 * treesPerGeneration

def MutateTrees(givenTrees, numToCreate):
    createdTrees = []
    
    totalScore = 0
    scoreArrForBinarySearch = []
    correspondingTree = []
    for tree in givenTrees:
        totalScore += tree.score
        scoreArrForBinarySearch.append(totalScore)
        correspondingTree.append(tree)
    
    for _ in range(numToCreate):
        numberInArr = random.uniform(0, totalScore)
        foundIndex = bisect.bisect_left(scoreArrForBinarySearch, numberInArr)
        chosenTree = correspondingTree[foundIndex].copy()

        nodesAsList = list(chosenTree.allNodes)
        nodeNum = random.randint(0, len(nodesAsList))
        theActualNode = nodesAsList[nodeNum]

        theActualNode.mutate()

        createdTrees.append(chosenTree)

    return createdTrees

def CrackedCanonicalRep(node):
    if node == None:
        return
    
    return (node.nodeName, node.left, node.right)
    

def ExploreTreeAndFindMatches(currentNode, treesSeenThusFarCount, treesToRootNodesInvolvedMapping, representationsToAddCurrentNodeTo):
    if(currentNode.type == NodeType.ACTION):
        return
    
    currentNodeRepresentation = CrackedCanonicalRep(currentNode)
    if currentNodeRepresentation not in treesSeenThusFarCount:
        treesSeenThusFarCount[currentNodeRepresentation] = 1
    else:
        treesSeenThusFarCount[currentNodeRepresentation] += 1
    
    for element in representationsToAddCurrentNodeTo:
        if element not in treesToRootNodesInvolvedMapping:
            representationsToAddCurrentNodeTo[element] = [currentNode]
        else:
            representationsToAddCurrentNodeTo[element].append(currentNode)

    ExploreTreeAndFindMatches(currentNode.left, treesSeenThusFarCount, treesToRootNodesInvolvedMapping, representationsToAddCurrentNodeTo)
    ExploreTreeAndFindMatches(currentNode.right, treesSeenThusFarCount, treesToRootNodesInvolvedMapping, representationsToAddCurrentNodeTo)

def ProtectCommonTrees(trees, protectCount):
    treesSeenThusFarCount = {}
    treesToNodesInvolvedMapping = {}

    for tree in trees:
        ExploreTreeAndFindMatches(tree.root, treesSeenThusFarCount, treesToNodesInvolvedMapping, [])

    for uniqueCombo in treesSeenThusFarCount.keys():
        if treesSeenThusFarCount[uniqueCombo] < protectCount:
            treesSeenThusFarCount.remove(uniqueCombo)
            treesToNodesInvolvedMapping.remove(uniqueCombo)
    
    return treesToNodesInvolvedMapping.Values

depthProbabilityMultiplier = 0.2

def UpdateDepths(currentNode, newDepth):
    if currentNode == None:
        return
    
    currentNode.depth = newDepth
    UpdateDepths(currentNode.left, newDepth + 1)
    UpdateDepths(currentNode.right, newDepth + 1)

def PerformCrossover(node1, node2):
    node1Parent = node1.parent
    node2Parent = node2.parent
    if(node1Parent.left == node1):
        if(node2Parent.left == node2):
            node1Parent.left = node2
            node2Parent.left = node1
        else:
            node1Parent.left = node2
            node2Parent.right = node1
    else:
        if(node2Parent.left == node2):
            node1Parent.right = node2
            node2Parent.left = node1
        else:
            node1Parent.right = node2
            node2Parent.right = node1

    UpdateDepths(node1, node2Parent.depth + 1)
    UpdateDepths(node2, node1Parent.depth + 1)

    return (node1.tree, node2.tree)

def CrossoverTrees(givenTrees, numToCreate, countToBeProtected):
    protectedNodes = ProtectCommonTrees(givenTrees, countToBeProtected)
    createdTrees = []

    for i in range(math.floor(numToCreate / 2)):
        chosenTreeA = random.choice(givenTrees).copy()

        chosenNodeA = None
        while True:
            chosenNodeA = random.choice(chosenTreeA.allNodes)
            if(chosenNodeA.type != NodeType.ACTION and chosenNodeA != chosenTreeA.root):
                if(CrackedCanonicalRep(chosenNodeA) in protectedNodes):
                    chanceOfAccepting = random.random()
                    if(chanceOfAccepting >= 0.5):
                        break
        
        chosenTreeB = chosenTreeA
        while chosenTreeA == chosenTreeB:
            chosenTreeB = random.choice(givenTrees).copy()

        chosenNodeB = None
        while True:
            chosenNodeB = random.choice(chosenTreeB.allNodes)
            if(chosenNodeB.type != NodeType.ACTION and chosenNodeB != chosenTreeB.root and CrackedCanonicalRep(chosenNodeA) != CrackedCanonicalRep(chosenNodeB)):
                if(CrackedCanonicalRep(chosenNodeB) in protectedNodes):
                    chanceOfAccepting = random.random()
                    if(chanceOfAccepting >= 0.5):
                        depth = abs(chosenNodeA.depth - chosenNodeB.depth)
                        chanceOfAccepting = random.random()
                        if(chanceOfAccepting >= depth * depthProbabilityMultiplier):
                            break        
        
        chosenTreeA, chosenTreeB = PerformCrossover(chosenNodeA, chosenNodeB)
        createdTrees.append(chosenTreeA)
        createdTrees.append(chosenTreeB)
    
    return createdTrees


countToBeProtected = 2

for i in range(0, numberGenerations, 1):
    for _ in range(treesPerGeneration - len(remainingTrees)):
        remainingTrees.append(BuildRandomTree(maxDepth, chanceOfEndingEarly, allConditions.copy(), allActions.copy()))
    
    for tree in remainingTrees:
        DoTreeUpkeep(tree)
        for _ in range(episodesPerTree):
            tree.score += RunGame(tree, game, episodesPerTree)
    
    remainingTrees = RankTreesByScore(remainingTrees)

    reproducedTrees = remainingTrees[:numTreesForReproduction]
    mutatedTrees = MutateTrees(reproducedTrees, numTreesForMutation)
    crossoverTrees = CrossoverTrees(reproducedTrees, numTreesForCrossover, countToBeProtected)

    remainingTrees = reproducedTrees + mutatedTrees + crossoverTrees

    # at halfway point, add up
    if(i == math.ceil(numberGenerations / 2)):
        countToBeProtected += 1

game.close()
remainingTrees = RankTreesByScore(remainingTrees)
for i in range(3):
    print("\n\nData about the {i}th best tree:")
    ShowTreeStructure(remainingTrees[i].root)